In [1]:
# Training Optimization Techniques

import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

SID4 = 8811
SEED = SID4

SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("SID4  =", SID4)
print("SEED  =", SEED)
print("SLICE =", SLICE)
print("HP_ID =", HP_ID)
print("CLS_A =", CLS_A)
print("CLS_B =", CLS_B)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Experiments will run on CPU.")

SID4  = 8811
SEED  = 8811
SLICE = 811
HP_ID = 3
CLS_A = 1
CLS_B = 0
Device: cuda
GPU: Tesla T4


In [2]:
from torch.utils.data import TensorDataset, DataLoader

# Fixed experiment settings
NUM_SAMPLES = 512
INPUT_SIZE = 28 * 28
NUM_CLASSES = 10
BATCH_SIZE = 32
TRAINING_STEPS = 20

# Create reproducible synthetic classification data
data_generator = torch.Generator().manual_seed(SEED)

features = torch.randn(
    NUM_SAMPLES,
    INPUT_SIZE,
    generator=data_generator
)

labels = torch.randint(
    low=0,
    high=NUM_CLASSES,
    size=(NUM_SAMPLES,),
    generator=data_generator
)

dataset = TensorDataset(features, labels)

data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Number of samples:", NUM_SAMPLES)
print("Input size:", INPUT_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Training steps:", TRAINING_STEPS)

Number of samples: 512
Input size: 784
Number of classes: 10
Batch size: 32
Training steps: 20


In [3]:
class OptimizationMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden_layers = nn.Sequential(
            nn.Linear(INPUT_SIZE, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        self.output_layer = nn.Linear(256, NUM_CLASSES)

    def forward(self, inputs, use_checkpoint=False):
        if use_checkpoint:
            hidden_output = torch.utils.checkpoint.checkpoint(
                self.hidden_layers,
                inputs,
                use_reentrant=False
            )
        else:
            hidden_output = self.hidden_layers(inputs)

        return self.output_layer(hidden_output)

In [4]:
test_model = OptimizationMLP().to(device)
test_batch, test_labels = next(iter(data_loader))

test_batch = test_batch.to(device)
test_labels = test_labels.to(device)

test_outputs = test_model(test_batch)

print("Input shape:", test_batch.shape)
print("Output shape:", test_outputs.shape)
print("Labels shape:", test_labels.shape)

Input shape: torch.Size([32, 784])
Output shape: torch.Size([32, 10])
Labels shape: torch.Size([32])


### Controlled Experiment Setup

A fixed synthetic classification dataset and the same multilayer perceptron model were created for all optimization experiments. The dataset contains 512 samples, each with 784 input features and one of 10 class labels. The batch size is fixed at 32, and each experiment will run for 20 training steps. Using the same model, data, batch size, and number of steps makes the comparison between optimization techniques more consistent.

In [5]:
def measure_tensor_creation(device_name, repetitions=20):
    """
    Measure the time required to create tensors on a device.
    """
    if device_name == "cuda" and not torch.cuda.is_available():
        return None

    if device_name == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for _ in range(repetitions):
        temporary_tensor = torch.randn(
            512,
            512,
            device=device_name
        )

    if device_name == "cuda":
        torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    return elapsed_time


cpu_creation_time = measure_tensor_creation("cpu")

print(f"CPU tensor creation time: {cpu_creation_time:.4f} seconds")

if torch.cuda.is_available():
    gpu_creation_time = measure_tensor_creation("cuda")
    print(f"GPU tensor creation time: {gpu_creation_time:.4f} seconds")
else:
    gpu_creation_time = None
    print("GPU tensor creation was skipped because no GPU is available.")

CPU tensor creation time: 0.0390 seconds
GPU tensor creation time: 0.0102 seconds


In [6]:
def initialize_model_weights(model, method="xavier"):
    """
    Initialize the linear layers using a selected method.
    """
    torch.manual_seed(SEED)

    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            if method == "xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif method == "kaiming":
                nn.init.kaiming_uniform_(
                    layer.weight,
                    nonlinearity="relu"
                )
            else:
                raise ValueError(
                    "method must be 'xavier' or 'kaiming'"
                )

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

    return model


xavier_model = OptimizationMLP().to(device)
kaiming_model = OptimizationMLP().to(device)

initialize_model_weights(xavier_model, method="xavier")
initialize_model_weights(kaiming_model, method="kaiming")

print("Xavier initialization completed.")
print("Kaiming initialization completed.")

Xavier initialization completed.
Kaiming initialization completed.


In [7]:
print(
    "Xavier weight mean:",
    xavier_model.hidden_layers[0].weight.mean().item()
)

print(
    "Xavier weight standard deviation:",
    xavier_model.hidden_layers[0].weight.std().item()
)

print(
    "Kaiming weight mean:",
    kaiming_model.hidden_layers[0].weight.mean().item()
)

print(
    "Kaiming weight standard deviation:",
    kaiming_model.hidden_layers[0].weight.std().item()
)

Xavier weight mean: -4.02969671995379e-05
Xavier weight standard deviation: 0.03933786228299141
Kaiming weight mean: -5.18104134243913e-05
Kaiming weight standard deviation: 0.050577253103256226


### Tensor Creation and Weight Initialization

Tensor creation was measured separately on the CPU and GPU to observe the cost of allocating data on each device. GPU tensor creation can be faster for large computational workloads, but the result depends on the available hardware and synchronization overhead. Xavier and Kaiming initialization produced controlled starting values for the model weights. The selected initialization method can influence the stability and speed of training because it determines the scale of the signals passing through the network at the beginning of training.